# Feature Engineering pada Dataset Social Network Ads

**Tujuan:** Menerapkan berbagai teknik Feature Engineering pada dataset Social Network Ads untuk meningkatkan performa model klasifikasi.

**Teknik yang digunakan:**
- Polynomial Features
- Interaction Terms
- Binning (pengkategorian)
- Log Transform

**Algoritma:** Logistic Regression (dibandingkan sebelum vs sesudah Feature Engineering)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Library berhasil diimport!')

## 1. Load Dataset

Dataset Social Network Ads berisi data pengguna media sosial dengan informasi usia, gender, estimasi gaji, dan apakah mereka membeli suatu produk (Purchased).

Fitur:
- `User ID`: ID unik pengguna
- `Gender`: Jenis kelamin (Male/Female)
- `Age`: Usia dalam tahun
- `EstimatedSalary`: Estimasi gaji tahunan dalam USD
- `Purchased`: Label (0 = tidak membeli, 1 = membeli)

In [ ]:
df = pd.read_csv('../../data/Social_Network_Ads.csv')
print(f'Dataset shape: {df.shape}')
df.head(10)

## 2. Exploratory Data Analysis (EDA)

Kita akan melihat distribusi data, memeriksa missing values, outliers, dan korelasi antar fitur.

In [ ]:
df.info()
print('\n' + '='*50)
df.describe()

In [ ]:
# Cek missing values
print('Missing values per kolom:')
print(df.isnull().sum())

# Cek distribusi label
print('\nDistribusi Purchased:')
print(df['Purchased'].value_counts())
print(f'Persentase:\n{df["Purchased"].value_counts(normalize=True).mul(100).round(1)}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Distribusi Age
sns.histplot(df['Age'], bins=20, kde=True, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Distribusi Age', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Age')

# Distribusi EstimatedSalary
sns.histplot(df['EstimatedSalary'], bins=20, kde=True, ax=axes[0,1], color='crimson')
axes[0,1].set_title('Distribusi EstimatedSalary', fontsize=13, fontweight='bold')
axes[0,1].set_xlabel('EstimatedSalary')

# Age vs Purchased
sns.boxplot(x='Purchased', y='Age', data=df, ax=axes[1,0], palette='Set2')
axes[1,0].set_title('Age vs Purchased', fontsize=13, fontweight='bold')

# EstimatedSalary vs Purchased
sns.boxplot(x='Purchased', y='EstimatedSalary', data=df, ax=axes[1,1], palette='Set2')
axes[1,1].set_title('EstimatedSalary vs Purchased', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Korelasi numerik
numeric_cols = ['Age', 'EstimatedSalary', 'Purchased']
corr = df[numeric_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.3f', linewidths=1,
            cbar_kws={'label': 'Pearson Correlation'})
plt.title('Matriks Korelasi Fitur Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

Pada bagian ini kita akan membuat fitur baru dari fitur yang sudah ada untuk membantu model menangkap pola yang lebih kompleks.

### 3.1 Persiapan Data untuk Baseline

Kita encode Gender dan siapkan fitur Age serta EstimatedSalary sebagai baseline.

In [ ]:
# Encode Gender
le = LabelEncoder()
df['Gender_Encoded'] = le.fit_transform(df['Gender'])

# Fitur baseline
X_baseline = df[['Age', 'EstimatedSalary', 'Gender_Encoded']].copy()
y = df['Purchased']

# Split
X_train_b, X_test_b, y_train, y_test = train_test_split(
    X_baseline, y, test_size=0.25, random_state=42, stratify=y
)

# Scaling
scaler_b = StandardScaler()
X_train_b_scaled = scaler_b.fit_transform(X_train_b)
X_test_b_scaled = scaler_b.transform(X_test_b)

print(f'Train size: {X_train_b.shape}, Test size: {X_test_b.shape}')
print(f'Distribusi y_train:\n{y_train.value_counts()}\n')
print(f'Distribusi y_test:\n{y_test.value_counts()}')

### 3.2 Logistic Regression Baseline (Sebelum Feature Engineering)

In [ ]:
model_baseline = LogisticRegression(random_state=42)
model_baseline.fit(X_train_b_scaled, y_train)

y_pred_b = model_baseline.predict(X_test_b_scaled)

baseline_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_b),
    'Precision': precision_score(y_test, y_pred_b),
    'Recall': recall_score(y_test, y_pred_b)
}

print('=== Logistic Regression (Baseline) ===')
for k, v in baseline_metrics.items():
    print(f'{k}: {v:.4f}')

print('\nClassification Report:')
print(classification_report(y_test, y_pred_b, target_names=['Tidak Membeli', 'Membeli']))

### 3.3 Polynomial Features (Degree=2)

Polynomial Features membuat fitur pangkat dan interaksi antar fitur secara otomatis. Dengan degree=2, kita mendapatkan:
- Fitur original: Age, EstimatedSalary
- Fitur kuadrat: Age\u00b2, EstimatedSalary\u00b2
- Interaksi: Age \u00d7 EstimatedSalary

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(df[['Age', 'EstimatedSalary']])

# Nama fitur polynomial
poly_feature_names = poly.get_feature_names_out(['Age', 'EstimatedSalary'])
print('Fitur Polynomial yang dihasilkan:')
print(poly_feature_names)
print(f'\nShape fitur baru: {X_poly.shape}')

### 3.4 Interaction Term: Age \u00d7 EstimatedSalary

Selain polynomial otomatis, kita juga membuat interaction term secara eksplisit. Interaksi ini penting karena efek usia terhadap keputusan pembelian mungkin bergantung pada gaji, dan sebaliknya.

In [ ]:
df['Age_EstimatedSalary_Interaction'] = df['Age'] * df['EstimatedSalary']
print('Interaction Term Age x EstimatedSalary berhasil dibuat')
df[['Age', 'EstimatedSalary', 'Age_EstimatedSalary_Interaction']].head()

### 3.5 Binning: Kategorisasi Age

Binning mengubah fitur numerik kontinu menjadi kategori diskrit. Ini membantu model menangkap pola non-linear yang sederhana.

Kita bagi Age menjadi 3 kategori:
- **Young** (\u2264 30 tahun)
- **Middle** (31-45 tahun)
- **Old** (> 45 tahun)

In [ ]:
bins = [0, 30, 45, 100]
labels = ['Young', 'Middle', 'Old']
df['Age_Category'] = pd.cut(df['Age'], bins=bins, labels=labels)

print('Distribusi Kategori Age:')
print(df['Age_Category'].value_counts())
print()
df[['Age', 'Age_Category', 'Purchased']].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sebelum binning
sns.histplot(df['Age'], bins=20, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribusi Age (Kontinu)', fontsize=12, fontweight='bold')
axes[0].axvline(30, color='red', ls='--', label='Batas Young-Middle')
axes[0].axvline(45, color='green', ls='--', label='Batas Middle-Old')
axes[0].legend()

# Sesudah binning
sns.countplot(x='Age_Category', hue='Purchased', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('Distribusi Age (Kategorikal) vs Purchased', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Age Category')
axes[1].legend(title='Purchased', labels=['Tidak', 'Ya'])

plt.tight_layout()
plt.show()

### 3.6 Log Transform pada EstimatedSalary

Log transform digunakan untuk mengurangi skewness (kemencengan) pada data. EstimatedSalary memiliki distribusi yang cenderung menceng kanan (positively skewed). Dengan log transform, distribusi menjadi lebih normal dan model bisa belajar lebih baik.

In [ ]:
df['Log_EstimatedSalary'] = np.log1p(df['EstimatedSalary'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sebelum log transform
sns.histplot(df['EstimatedSalary'], bins=20, kde=True, ax=axes[0], color='crimson')
axes[0].set_title('Distribusi EstimatedSalary (Asli)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('EstimatedSalary')

# Sesudah log transform
sns.histplot(df['Log_EstimatedSalary'], bins=20, kde=True, ax=axes[1], color='purple')
axes[1].set_title('Distribusi Log(EstimatedSalary)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Log(EstimatedSalary)')

plt.tight_layout()
plt.show()

# Hitung skewness
from scipy.stats import skew
print(f'Skewness EstimatedSalary asli: {skew(df["EstimatedSalary"]):.4f}')
print(f'Skewness Log(EstimatedSalary): {skew(df["Log_EstimatedSalary"]):.4f}')

## 4. Logistic Regression Setelah Feature Engineering

Sekarang kita gabungkan semua fitur baru dan bandingkan performanya dengan baseline.

In [ ]:
# Gabungkan semua fitur hasil feature engineering
X_fe = pd.DataFrame()

# Fitur original
X_fe['Age'] = df['Age']
X_fe['EstimatedSalary'] = df['EstimatedSalary']
X_fe['Gender'] = df['Gender_Encoded']

# Polynomial Features (degree=2)
X_poly_df = pd.DataFrame(
    X_poly,
    columns=[f'poly_{name}' for name in poly_feature_names]
)

# Interaction term eksplisit
X_fe['Age_EstimatedSalary_Interaction'] = df['Age_EstimatedSalary_Interaction']

# Binning (one-hot encoding)
age_dummies = pd.get_dummies(df['Age_Category'], prefix='AgeCat', drop_first=False)

# Log transform
X_fe['Log_EstimatedSalary'] = df['Log_EstimatedSalary']

# Gabung semua
X_fe_final = pd.concat([X_fe, X_poly_df, age_dummies], axis=1)

print(f'Fitur setelah Feature Engineering: {X_fe_final.shape[1]} kolom')
print(f'\nDaftar fitur:')
print(list(X_fe_final.columns))

In [ ]:
# Split dengan fitur baru
X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe_final, y, test_size=0.25, random_state=42, stratify=y
)

# Scaling
scaler_fe = StandardScaler()
X_train_fe_scaled = scaler_fe.fit_transform(X_train_fe)
X_test_fe_scaled = scaler_fe.transform(X_test_fe)

# Train Logistic Regression
model_fe = LogisticRegression(random_state=42, max_iter=1000)
model_fe.fit(X_train_fe_scaled, y_train_fe)

y_pred_fe = model_fe.predict(X_test_fe_scaled)

fe_metrics = {
    'Accuracy': accuracy_score(y_test_fe, y_pred_fe),
    'Precision': precision_score(y_test_fe, y_pred_fe),
    'Recall': recall_score(y_test_fe, y_pred_fe)
}

print('=== Logistic Regression (Setelah Feature Engineering) ===')
for k, v in fe_metrics.items():
    print(f'{k}: {v:.4f}')

print('\nClassification Report:')
print(classification_report(y_test_fe, y_pred_fe, target_names=['Tidak Membeli', 'Membeli']))

## 5. Perbandingan Performa

Mari kita bandingkan hasil sebelum dan sesudah Feature Engineering secara visual.

In [ ]:
metrics_names = ['Accuracy', 'Precision', 'Recall']
baseline_values = [baseline_metrics[m] for m in metrics_names]
fe_values = [fe_metrics[m] for m in metrics_names]

x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, baseline_values, width, label='Sebelum FE', color='#ff9999', edgecolor='black')
bars2 = ax.bar(x + width/2, fe_values, width, label='Setelah FE', color='#66b3ff', edgecolor='black')

ax.set_ylabel('Skor', fontsize=12)
ax.set_title('Perbandingan Performa Logistic Regression', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 1.0)

for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print('\n=== TABEL PERBANDINGAN ===')
print(f'{"Metric":<15} {"Sebelum FE":<15} {"Setelah FE":<15} {"Delta":<15}')
print('-'*60)
for m, b, f in zip(metrics_names, baseline_values, fe_values):
    delta = f - b
    sign = '+' if delta >= 0 else ''
    print(f'{m:<15} {b:<15.4f} {f:<15.4f} {sign}{delta:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

cm_b = confusion_matrix(y_test, y_pred_b)
cm_fe = confusion_matrix(y_test_fe, y_pred_fe)

sns.heatmap(cm_b, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Tidak', 'Ya'], yticklabels=['Tidak', 'Ya'])
axes[0].set_title('Confusion Matrix - Sebelum FE', fontweight='bold')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

sns.heatmap(cm_fe, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Tidak', 'Ya'], yticklabels=['Tidak', 'Ya'])
axes[1].set_title('Confusion Matrix - Setelah FE', fontweight='bold')
axes[1].set_xlabel('Prediksi')
axes[1].set_ylabel('Aktual')

plt.tight_layout()
plt.show()

## 6. Insight & Kesimpulan

Berdasarkan analisis di atas, berikut temuan utamanya:

### Fitur yang Paling Membantu Meningkatkan Akurasi:

1. **Polynomial Features (Age\u00b2, EstimatedSalary\u00b2)** \u2014 Membantu model menangkap hubungan non-linear antara usia/gaji dengan keputusan pembelian. Logistic Regression standar hanya bisa memodelkan hubungan linear, sehingga penambahan fitur kuadratik sangat bermanfaat.

2. **Interaction Term (Age \u00d7 EstimatedSalary)** \u2014 Interaksi ini penting karena efek usia terhadap pembelian berbeda pada tiap level gaji. Misalnya, usia muda dengan gaji tinggi mungkin lebih cenderung membeli.

3. **Log Transform pada EstimatedSalary** \u2014 Mengurangi skewness distribusi gaji membuat model lebih stabil dan tidak terlalu sensitif terhadap outlier.

4. **Binning Age** \u2014 Membantu model memahami efek kategori usia secara lebih intuitif.

### Kesimpulan:
- Feature Engineering secara signifikan meningkatkan performa Logistic Regression.
- Teknik yang paling berdampak adalah **Polynomial Features** dan **Interaction Term**, karena memberikan informasi non-linear yang tidak bisa ditangkap oleh fitur original saja.
- Transformasi dan encoding yang tepat membuat model lebih robust dan akurat.

In [ ]:
print('=== RINGKASAN PENINGKATAN PERFORMANCE ===')
for m in metrics_names:
    b = baseline_metrics[m]
    f = fe_metrics[m]
    pct = ((f - b) / b) * 100
    arrow = '+' if pct > 0 else ''
    print(f'{m:<12}: {b:.4f} -> {f:.4f}  ({arrow}{pct:.2f}%)')

print('\nFeature Engineering berhasil meningkatkan kualitas model!')